In [1]:
from src import *
import numpy as np
import pandas as pd
import re

from bs4 import BeautifulSoup

In [2]:
def extractText(regex, text, G=0):
    try:
        return re.search(regex, text).group(G)
    except:
        return ''

In [3]:
def Parse_VA(soup : BeautifulSoup):
    output = []
    
    for tr in soup.find('table', attrs={'class':"js-table-people-character table-people-character"}).find_all('tr'):
        res = dict()
    
        td = tr.find_all('td',  attrs={'class':'borderClass'}) 
    
        
        a = td[1].find('a', attrs={'class':'js-people-title'})
        res['Title'] = a.text 
        res['Link'] = a['href'] 
        res['ID'] = int(re.search(r'/anime/(\d+)', a['href']).group(1))
    
        text = td[1].find('div', attrs={'class':'spaceit_pad anime-info-text'}).text
        res['Type'] = extractText(r'^[^,]+', text)
        res['Season'] = extractText(r'(?<=,)(.*?)(?=\s\d)', text).strip()
        res['Year'] =  extractText(r'(\d+)$', text)
    
    
        a = td[2].find('a')
        res['Character'] = a.text 
        res['CharacterLink'] = a['href'] 
        res['CharacterID'] = int(re.search(r'/character/(\d+)', a['href']).group(1))
        res['CharacterRole'] = td[2].find_all('div', attrs={'class':'spaceit_pad'})[1].text.strip()
        res['CharacterFavorites'] = int(td[2].find('div', attrs={'class':'spaceit_pad character-total-favorites'}).text.split()[0].replace(',',''))
    
        output.append(res)
    
    return output

In [4]:
def Parse_Staff(soup : BeautifulSoup):
    output = []
    for tr in soup.find('table', attrs={'class':"js-table-people-staff"}).find_all('tr'):
        res = dict()
    
        a = tr.find('a', attrs={'class':'js-people-title'})
        res['Title'] = a.text 
        res['Link'] = a['href'] 
        res['ID'] = int(re.search(r'/anime/(\d+)', a['href']).group(1))
         
        res['Roles'] = tr.find('a', attrs={'class':'Lightbox_AddEdit button_add ga-click addtolist'}).find_next_sibling().text
    
        text = tr.find('div', attrs={'class':'spaceit_pad anime-info-text'}).text
        res['Type'] = extractText(r'^[^,]+', text)
        res['Season'] = extractText(r'(?<=,)(.*?)(?=\s\d)', text).strip()
    
        try:
            res['Year'] = int(extractText(r'(\d+)$', text))
        except:
            res['Year'] = np.nan
    
        try:
            res['Score'] = float(tr.find('span', attrs={'class':'score-val'}).text)
        except:
            res['Score'] = np.nan
        
        try:
            res['Members'] = float(tr.find('div', attrs={'class':'spaceit_pad series-total-members'}).text.replace(',', '').split()[0])
        except:
            res['Members'] = np.nan
        
        output.append(res)

    return output

In [5]:
#RANGE = np.concatenate([np.arange(759,1000), np.arange(2000,3000)])
RANGE = np.arange(1, 10)
#RANGE = np.arange(2150,2200)

In [6]:
for PERSONNEL_ID in RANGE:
    URL = f'https://myanimelist.net/people/{PERSONNEL_ID}'
    
    Scraper.MINIMUM_DELAY = np.random.randint(15, 30)
    
    try:
        soup = Scraper().scrape(URL)
        PERSONNEL_NAME = soup.find('h1', attrs={'class':'title-name h1_bold_none'}).text
        PERSONNEL_NAME = re.sub(r'[^a-zA-Z0-9\u00C0-\u017F\u0400-\u04FF\u0600-\u06FF\u4e00-\u9fff\s]', '', PERSONNEL_NAME)
    except Exception as e:
        print(f"\033[91mNo page for {PERSONNEL_ID}: {e}\033[0m")
        continue
    
    try:
        #pd.DataFrame(Parse_VA(soup)).set_index('ID').to_csv(f'Roles/VA/{PERSONNEL_ID}_{PERSONNEL_NAME}.csv')
        pd.DataFrame(Parse_VA(soup)).set_index('ID').to_csv('t0.csv')
        print(f'[VA]\t {PERSONNEL_ID}: {PERSONNEL_NAME}')
    except Exception as e:
        print(f"\033[91m[VA]\t{PERSONNEL_ID}: {PERSONNEL_NAME}: {e}\033[0m")
    
    try:
        #pd.DataFrame(Parse_Staff(soup)).set_index('ID').to_csv(f'Roles/Staff/{PERSONNEL_ID}_{PERSONNEL_NAME}.csv')
        pd.DataFrame(Parse_Staff(soup)).set_index('ID').to_csv('t1.csv')
        print(f'[Staff]\t {PERSONNEL_ID}: {PERSONNEL_NAME}')
    except Exception as e:
        print(f"\033[91m[Staff]\t{PERSONNEL_ID}: {PERSONNEL_NAME}: {e}\033[0m")    

{'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36'}
[VA]	 1: Seki Tomokazu
[Staff]	 1: Seki Tomokazu
{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36'}
[VA]	 2: Sugita Tomokazu
[Staff]	 2: Sugita Tomokazu
{'User-Agent': 'Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36'}
[VA]	 3: Yukino Satsuki
[Staff]	 3: Yukino Satsuki
{'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0'}
[VA]	 4: Hirano Aya
[Staff]	 4: Hirano Aya
{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36'}
[VA]	 5: Suzumura Kenichi
[Staff]	 5: Suzumura Kenichi
{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 OPR/116